# Detector de objetos para o site de assistência (MobileNet, Colab)

Treina um detector **SSDLite + MobileNetV3-Large** (pré-treinado no COCO) com objetos da lista
`objetos_para_treino.csv` e exporta em **ONNX** para rodar no navegador do celular, no site
`ohtopico.com.br/teste-mestrado`.

**Não precisa de conta nem de marcar fotos.** O notebook baixa sozinho, do
[Open Images V7](https://storage.googleapis.com/openimages/web/index.html) (dataset aberto do Google, licença CC BY),
fotos que já vêm com as caixas marcadas para 49 objetos da lista que o site ainda não reconhece:
porta, maçaneta, escada, lixeira, torneira, fogão, moeda, óculos, cadeira de rodas, poste, árvore e outros.

**Como rodar**
1. Ambiente de execução > Alterar o tipo > **GPU** (T4 ou melhor).
2. Ambiente de execução > **Executar tudo** e autorize o Google Drive quando pedir.

A primeira vez demora: o download leva uns 30 a 60 minutos e o treino algumas horas. As fotos já preparadas ficam
salvas no Drive (`dataset_open_images.zip`), então nas próximas vezes o notebook pula o download.

**Segunda rodada (foco):** com `FOCO` preenchido na configuração (já vem com os 12 objetos fracos da primeira
rodada), o notebook baixa mais fotos deles, sorteia essas fotos 3 vezes mais no treino e começa dos pesos da primeira
rodada (`melhor.pt`), com 20 épocas. Tudo é salvo em `detector_assistencia/foco/`, sem apagar a primeira rodada.
Para voltar à rodada normal, deixe `FOCO = {}`.

Objetos que **não existem** no Open Images (chave, carteira, cédula, bengala, extintor, piso tátil, corrimão…)
vão precisar de fotos suas marcadas; isso fica para uma segunda etapa e não é necessário agora.

In [ ]:
!nvidia-smi -L
!pip install -q torchmetrics pycocotools onnx onnxruntime

## 1. Configuração

In [ ]:
# Onde guardar tudo no Drive (fotos preparadas, pesos, métricas e o modelo para o site)
PASTA_DRIVE = "/content/drive/MyDrive/Mestrado/detector_assistencia"

# Classe do Open Images -> nome que o site vai falar (o "_" vira espaço; pode ter acento).
# Para tirar um objeto, apague a linha. Duas classes com o mesmo nome viram um objeto só.
OBJETOS = {
    "Door": "porta", "Door handle": "maçaneta", "Stairs": "escada", "Window": "janela",
    "Waste container": "lixeira", "Street light": "poste", "Tree": "árvore", "Wheelchair": "cadeira_de_rodas",
    "Coin": "moeda", "Glasses": "óculos", "Sunglasses": "óculos_de_sol", "Headphones": "fone_de_ouvido",
    "Watch": "relógio_de_pulso", "Pen": "caneta", "Footwear": "sapato", "Plastic bag": "sacola",
    "Plate": "prato", "Frying pan": "frigideira", "Kettle": "chaleira", "Coffeemaker": "cafeteira",
    "Blender": "liquidificador", "Gas stove": "fogão", "Tap": "torneira", "Cutting board": "tábua_de_corte",
    "Tin can": "lata", "Milk": "caixa_de_leite", "Bread": "pão", "Washing machine": "máquina_de_lavar",
    "Coffee table": "mesa_de_centro", "Bookcase": "estante", "Shelf": "prateleira", "Lamp": "luminária",
    "Ceiling fan": "ventilador_de_teto", "Mechanical fan": "ventilador", "Light switch": "interruptor",
    "Power plugs and sockets": "tomada", "Curtain": "cortina", "Mirror": "espelho",
    "Pillow": "travesseiro", "Wardrobe": "guarda-roupa", "Chest of drawers": "cômoda", "Nightstand": "mesa_de_cabeceira",
    "Shower": "chuveiro", "Towel": "toalha", "Soap dispenser": "saboneteira", "Toilet paper": "papel_higiênico",
    "Chicken": "galinha", "Mouse": "rato", "Insect": "inseto",
}
FOTOS_POR_OBJETO = 300         # de treino; mais fotos = modelo melhor, download e treino mais longos

# Segunda rodada, focada no que ficou fraco na primeira (AP baixo em ap_por_objeto.csv):
# esses objetos ganham mais fotos e aparecem 3 vezes mais no treino. Deixe FOCO = {} para a rodada normal.
FOCO = {"porta": 1500, "escada": 1500, "janela": 1200, "torneira": 1200, "poste": 1200, "chuveiro": 1200,
        "fogão": 1000, "sapato": 1000, "árvore": 800, "óculos_de_sol": 800, "saboneteira": 800, "prateleira": 800}
PESO_FOCO = 3
RODADA = "foco" if FOCO else "normal"          # cada rodada salva numa subpasta, sem apagar a anterior
# Começa dos pesos da rodada anterior em vez do zero: aprende mais rápido (None = começar do COCO)
PESOS_INICIAIS = "/content/drive/MyDrive/Mestrado/detector_assistencia/melhor.pt" if FOCO else None
FOTOS_VALIDACAO = 40           # por objeto, para validação e para teste
LADO_MAXIMO = 640              # as fotos são reduzidas para isso (mais rápido e ocupa menos)

# Opcional, para depois: uma pasta própria em COCO JSON (subpastas train/, valid/ e test/,
# cada uma com _annotations.coco.json). Se preencher, o Open Images não é usado.
DATASET_DIR = None

EPOCAS = 20 if PESOS_INICIAIS else 30
LOTE = 32
LR = 0.005 if PESOS_INICIAIS else 0.015   # para LOTE=32; partindo de um modelo já treinado, passo menor
TAMANHO = 320                  # entrada do SSDLite
SEMENTE = 0

## 2. Drive e dataset

In [ ]:
import os, io, json, shutil, zipfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from google.colab import drive
drive.mount("/content/drive")
PASTA_SAIDA = PASTA_DRIVE if RODADA == "normal" else f"{PASTA_DRIVE}/{RODADA}"
os.makedirs(PASTA_SAIDA, exist_ok=True)
ZIP_DRIVE = f"{PASTA_DRIVE}/dataset_open_images{'' if RODADA == 'normal' else '_' + RODADA}.zip"

# Arquivos públicos do Open Images: lista de classes, caixas marcadas e as fotos
GCS = "https://storage.googleapis.com/openimages"
CAIXAS = {"train": f"{GCS}/v6/oidv6-train-annotations-bbox.csv",       # 2,3 GB, leva alguns minutos
          "validation": f"{GCS}/v5/validation-annotations-bbox.csv",
          "test": f"{GCS}/v5/test-annotations-bbox.csv"}
FOTO = "https://s3.amazonaws.com/open-images-dataset/{split}/{id}.jpg"


def baixar_open_images(split_oi, por_objeto, pasta):
    """Escolhe até por_objeto fotos de cada classe, baixa, reduz e grava em COCO JSON com os nomes em português."""
    classes = pd.read_csv(f"{GCS}/v7/oidv7-class-descriptions-boxable.csv")
    mid_para_nome = {m: OBJETOS[n] for m, n in zip(classes.LabelName, classes.DisplayName) if n in OBJETOS}
    faltando = set(OBJETOS) - set(classes.DisplayName[classes.LabelName.isin(mid_para_nome)])
    assert not faltando, f"classes que não existem no Open Images: {faltando}"

    colunas = ["ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax", "IsDepiction"]
    caixas = pd.concat(p[p.LabelName.isin(mid_para_nome)]
                       for p in pd.read_csv(CAIXAS[split_oi], usecols=colunas, chunksize=2_000_000))
    caixas["nome"] = caixas.LabelName.map(mid_para_nome)
    # Sorteia as fotos de cada objeto (sem desenhos), e depois pega TODAS as caixas da lista nessas fotos
    reais = caixas[caixas.IsDepiction != 1].drop_duplicates(["ImageID", "nome"])
    sorteadas = reais.sample(frac=1, random_state=SEMENTE)
    # No treino, os objetos do FOCO ganham mais fotos; validação e teste ficam iguais, para comparar as rodadas
    limite = sorteadas.nome.map(lambda n: FOCO.get(n, por_objeto) if split_oi == "train" else por_objeto)
    escolhidas = set(sorteadas[sorteadas.groupby("nome").cumcount() < limite].ImageID)
    caixas = caixas[caixas.ImageID.isin(escolhidas)]
    print(split_oi, ":", len(escolhidas), "fotos,", len(caixas), "caixas")
    print(caixas.drop_duplicates(["ImageID", "nome"]).nome.value_counts().to_string())

    pasta.mkdir(parents=True, exist_ok=True)
    def baixar(id_foto):
        destino = pasta / f"{id_foto}.jpg"
        try:
            if not destino.exists():
                dados = urllib.request.urlopen(FOTO.format(split=split_oi, id=id_foto), timeout=30).read()
                img = ImageOps.exif_transpose(Image.open(io.BytesIO(dados))).convert("RGB")
                img.thumbnail((LADO_MAXIMO, LADO_MAXIMO))
                img.save(destino, quality=90)
            with Image.open(destino) as img:
                return id_foto, img.size
        except Exception as erro:
            print("falhou", id_foto, erro)
            return id_foto, None
    with ThreadPoolExecutor(32) as pool:
        tamanhos = {i: t for i, t in pool.map(baixar, sorted(escolhidas)) if t}

    nomes = sorted(set(OBJETOS.values()))
    id_cat = {n: i + 1 for i, n in enumerate(nomes)}
    id_img = {i: k + 1 for k, i in enumerate(sorted(tamanhos))}
    imagens = [{"id": id_img[i], "file_name": f"{i}.jpg", "width": w, "height": h} for i, (w, h) in tamanhos.items()]
    anotacoes = []
    for c in caixas[caixas.ImageID.isin(tamanhos)].itertuples():
        w, h = tamanhos[c.ImageID]
        x, y, bw, bh = c.XMin * w, c.YMin * h, (c.XMax - c.XMin) * w, (c.YMax - c.YMin) * h
        anotacoes.append({"id": len(anotacoes) + 1, "image_id": id_img[c.ImageID], "category_id": id_cat[c.nome],
                          "bbox": [x, y, bw, bh], "area": bw * bh, "iscrowd": 0})
    coco = {"images": imagens, "annotations": anotacoes, "categories": [{"id": i, "name": n} for n, i in id_cat.items()]}
    (pasta / "_annotations.coco.json").write_text(json.dumps(coco, ensure_ascii=False))
    print(pasta.name, ":", len(imagens), "fotos salvas,", len(anotacoes), "caixas")


if DATASET_DIR is None:
    DATASET_DIR = "/content/dataset"
    if os.path.exists(ZIP_DRIVE):
        print("Usando as fotos já preparadas no Drive")
        zipfile.ZipFile(ZIP_DRIVE).extractall(DATASET_DIR)
    else:
        for split_oi, split, n in (("train", "train", FOTOS_POR_OBJETO),
                                   ("validation", "valid", FOTOS_VALIDACAO),
                                   ("test", "test", FOTOS_VALIDACAO)):
            baixar_open_images(split_oi, n, Path(DATASET_DIR) / split)
        shutil.make_archive(ZIP_DRIVE[:-4], "zip", DATASET_DIR)
print("dataset em", DATASET_DIR, "->", os.listdir(DATASET_DIR))

In [ ]:
import json, random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision import tv_tensors
from torchvision.transforms import v2 as T

random.seed(SEMENTE); np.random.seed(SEMENTE); torch.manual_seed(SEMENTE)

def ler_coco(split):
    return json.loads((Path(DATASET_DIR) / split / "_annotations.coco.json").read_text())

# Algumas ferramentas criam uma categoria "mãe" sem nenhuma caixa: só entram categorias usadas.
# As classes ficam numeradas de 1 a K (0 é o fundo, como o torchvision espera).
usadas = set()
for split in ("train", "valid", "test"):
    if (Path(DATASET_DIR) / split).exists():
        usadas |= {a["category_id"] for a in ler_coco(split)["annotations"]}
categorias = sorted((c for c in ler_coco("train")["categories"] if c["id"] in usadas), key=lambda c: c["name"])
CLASSES = [c["name"] for c in categorias]
ID_PARA_INDICE = {c["id"]: i + 1 for i, c in enumerate(categorias)}
print(len(CLASSES), "classes:", CLASSES)


class DatasetCOCO(torch.utils.data.Dataset):
    def __init__(self, split, transform):
        coco = ler_coco(split)
        self.pasta = Path(DATASET_DIR) / split
        self.imagens = coco["images"]
        self.caixas = {}
        for a in coco["annotations"]:
            if a["category_id"] in ID_PARA_INDICE and a["bbox"][2] > 1 and a["bbox"][3] > 1:
                self.caixas.setdefault(a["image_id"], []).append(a)
        self.transform = transform

    def __len__(self):
        return len(self.imagens)

    def __getitem__(self, i):
        info = self.imagens[i]
        img = Image.open(self.pasta / info["file_name"]).convert("RGB")
        anns = self.caixas.get(info["id"], [])
        xywh = torch.tensor([a["bbox"] for a in anns], dtype=torch.float32).reshape(-1, 4)
        caixas = tv_tensors.BoundingBoxes(xywh, format="XYWH", canvas_size=(img.height, img.width))
        alvo = {"boxes": T.ConvertBoundingBoxFormat("XYXY")(caixas),
                "labels": torch.tensor([ID_PARA_INDICE[a["category_id"]] for a in anns], dtype=torch.int64)}
        return self.transform(tv_tensors.Image(T.functional.pil_to_tensor(img)), alvo)


# Receita padrão do SSD: cor, zoom para fora, recorte e espelhamento horizontal
# (espelhar não atrapalha: o lado é calculado depois, pela posição na imagem)
treino_tf = T.Compose([
    T.RandomPhotometricDistort(p=0.8),
    T.RandomZoomOut(fill={tv_tensors.Image: (124, 116, 104), "others": 0}, side_range=(1.0, 3.0), p=0.5),
    T.RandomIoUCrop(),
    T.RandomHorizontalFlip(),
    T.ToDtype(torch.float32, scale=True),
    T.SanitizeBoundingBoxes(),
])
avaliacao_tf = T.Compose([T.ToDtype(torch.float32, scale=True)])

juntar = lambda lote: tuple(zip(*lote))
treino = DatasetCOCO("train", treino_tf)
validacao = DatasetCOCO("valid", avaliacao_tf)
teste = DatasetCOCO("test", avaliacao_tf) if (Path(DATASET_DIR) / "test").exists() else validacao
import os  # usa todos os núcleos do Colab para preparar as fotos (é o que limita a velocidade)
carregar = lambda ds, emb: torch.utils.data.DataLoader(ds, batch_size=LOTE, shuffle=emb, num_workers=os.cpu_count(), persistent_workers=True, collate_fn=juntar, drop_last=emb and len(ds) > LOTE)
dl_treino, dl_val, dl_teste = carregar(treino, True), carregar(validacao, False), carregar(teste, False)
if FOCO:
    # Fotos com algum objeto do FOCO são sorteadas PESO_FOCO vezes mais em cada época
    pesos = [PESO_FOCO if any(CLASSES[ID_PARA_INDICE[a["category_id"]] - 1] in FOCO for a in treino.caixas.get(info["id"], []))
             else 1.0 for info in treino.imagens]
    sorteio = torch.utils.data.WeightedRandomSampler(pesos, num_samples=len(pesos), replacement=True)
    dl_treino = torch.utils.data.DataLoader(treino, batch_size=LOTE, sampler=sorteio, num_workers=os.cpu_count(),
                                            persistent_workers=True, collate_fn=juntar, drop_last=len(treino) > LOTE)
    print(f"{sum(p > 1 for p in pesos)} fotos com objetos do foco, sorteadas {PESO_FOCO}x mais")
print(len(treino), "treino |", len(validacao), "validação |", len(teste), "teste")

## 3. Modelo

SSDLite320 com backbone MobileNetV3-Large pré-treinado no COCO. A cabeça de classificação é trocada para as
suas classes; a de regressão (onde estão as caixas) continua com os pesos do COCO.

In [ ]:
from functools import partial
from torch import nn
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection import _utils as det_utils
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

def criar_modelo(num_classes, pre_treinado=True):
    # Sem pré-treino (na exportação os pesos vêm do melhor.pt) não baixa nada
    modelo = (ssdlite320_mobilenet_v3_large(weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1) if pre_treinado
              else ssdlite320_mobilenet_v3_large(weights=None, weights_backbone=None))
    canais = det_utils.retrieve_out_channels(modelo.backbone, (TAMANHO, TAMANHO))
    ancoras = modelo.anchor_generator.num_anchors_per_location()
    modelo.head.classification_head = SSDLiteClassificationHead(
        canais, ancoras, num_classes, partial(nn.BatchNorm2d, eps=0.001, momentum=0.03))
    return modelo

modelo = criar_modelo(len(CLASSES) + 1).to(DISPOSITIVO)
if PESOS_INICIAIS:
    modelo.load_state_dict(torch.load(PESOS_INICIAIS, map_location=DISPOSITIVO))
    print("Começando dos pesos de", PESOS_INICIAIS)
print(sum(p.numel() for p in modelo.parameters()) / 1e6, "milhões de parâmetros")

## 4. Treino

In [ ]:
import math, time
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def avaliar(modelo, dl, por_classe=False):
    modelo.eval()
    mapa = MeanAveragePrecision(iou_type="bbox", class_metrics=por_classe)
    mapa.warn_on_many_detections = False
    for imagens, alvos in dl:
        saidas = modelo([im.to(DISPOSITIVO) for im in imagens])
        mapa.update([{k: v.cpu() for k, v in s.items()} for s in saidas],
                    [{"boxes": a["boxes"].as_subclass(torch.Tensor), "labels": a["labels"]} for a in alvos])
    return mapa.compute()

parametros = [p for p in modelo.parameters() if p.requires_grad]
otimizador = torch.optim.SGD(parametros, lr=LR, momentum=0.9, weight_decay=4e-5)
passos = EPOCAS * max(1, len(dl_treino))
aquecimento = min(500, passos // 10)
agendador = torch.optim.lr_scheduler.LambdaLR(otimizador, lambda s: (s + 1) / aquecimento if s < aquecimento
                                               else 0.5 * (1 + math.cos(math.pi * (s - aquecimento) / max(1, passos - aquecimento))))
escala = torch.amp.GradScaler(enabled=DISPOSITIVO == "cuda")

historico, melhor = [], -1.0
for epoca in range(1, EPOCAS + 1):
    modelo.train()
    inicio, soma = time.time(), 0.0
    for imagens, alvos in dl_treino:
        imagens = [im.to(DISPOSITIVO) for im in imagens]
        alvos = [{"boxes": a["boxes"].as_subclass(torch.Tensor).to(DISPOSITIVO), "labels": a["labels"].to(DISPOSITIVO)} for a in alvos]
        with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
            perdas = modelo(imagens, alvos)
            perda = sum(perdas.values())
        otimizador.zero_grad(set_to_none=True)
        escala.scale(perda).backward()
        escala.step(otimizador)
        escala.update()
        agendador.step()
        soma += perda.item()
    m = avaliar(modelo, dl_val)
    linha = {"epoca": epoca, "perda": soma / max(1, len(dl_treino)), "mAP": m["map"].item(), "mAP50": m["map_50"].item()}
    historico.append(linha)
    if linha["mAP50"] > melhor:
        melhor = linha["mAP50"]
        torch.save(modelo.state_dict(), f"{PASTA_SAIDA}/melhor.pt")
    print(f"época {epoca:3d} | perda {linha['perda']:.3f} | mAP {linha['mAP']:.3f} | mAP50 {linha['mAP50']:.3f} | {time.time() - inicio:.0f}s")

import pandas as pd
pd.DataFrame(historico).to_csv(f"{PASTA_SAIDA}/historico.csv", index=False)

## 5. Resultado no teste (por objeto)

In [ ]:
modelo.load_state_dict(torch.load(f"{PASTA_SAIDA}/melhor.pt", map_location=DISPOSITIVO))
m = avaliar(modelo, dl_teste, por_classe=True)
print(f"teste: mAP {m['map']:.3f} | mAP50 {m['map_50']:.3f}")
por_classe = pd.DataFrame({"objeto": [CLASSES[int(c) - 1] for c in m["classes"].reshape(-1)],
                           "AP": m["map_per_class"].reshape(-1).numpy()}).sort_values("AP")
por_classe.to_csv(f"{PASTA_SAIDA}/ap_por_objeto.csv", index=False)
por_classe  # os de AP mais baixo precisam de mais fotos

In [ ]:
import matplotlib.pyplot as plt
modelo.eval()
fig, eixos = plt.subplots(2, 3, figsize=(15, 9))
for eixo, i in zip(eixos.flat, random.sample(range(len(teste)), min(6, len(teste)))):
    img, _ = teste[i]
    with torch.no_grad():
        s = modelo([img.to(DISPOSITIVO)])[0]
    eixo.imshow(img.permute(1, 2, 0).numpy()); eixo.axis("off")
    for caixa, rot, pont in zip(s["boxes"].cpu(), s["labels"].cpu(), s["scores"].cpu()):
        if pont < 0.4: continue
        x0, y0, x1, y1 = caixa.tolist()
        eixo.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="#166c40", lw=2))
        eixo.text(x0, y0, f"{CLASSES[rot - 1]} {pont:.2f}", color="white", backgroundcolor="#166c40", fontsize=9)
plt.tight_layout()

## 6. Exportar para o site

Gera `objetos.onnx` e `objetos_labels.txt` em `modelo_site/`, dentro da pasta da rodada (`detector_assistencia/foco/` na rodada focada). Copie os dois para a pasta
`teste-mestrado/modelos/` do repositório do site: ele passa a usar este detector junto com o que já tinha
(os 80 objetos do COCO), sem mudar nada no código. A saída do ONNX já vem filtrada (NMS incluído): caixas em
pixels da imagem de entrada, pontuações e rótulos.

In [ ]:
import onnxruntime as ort

modelo_cpu = criar_modelo(len(CLASSES) + 1, pre_treinado=False)
modelo_cpu.load_state_dict(torch.load(f"{PASTA_SAIDA}/melhor.pt", map_location="cpu"))
modelo_cpu.score_thresh = 0.3          # abaixo disso o site nem recebe a caixa
modelo_cpu.detections_per_img = 20
modelo_cpu.eval()

saida = Path(PASTA_SAIDA) / "modelo_site"
saida.mkdir(exist_ok=True)
exemplo = torch.rand(1, 3, TAMANHO, TAMANHO)
torch.onnx.export(modelo_cpu, (exemplo,), saida / "objetos.onnx", opset_version=17,
                  input_names=["imagem"], output_names=["caixas", "pontuacoes", "rotulos"], dynamo=False)
(saida / "objetos_labels.txt").write_text("\n".join(CLASSES) + "\n", encoding="utf-8")

# Confere que o ONNX dá o mesmo resultado que o PyTorch numa imagem de teste
img, _ = teste[0]
img = torch.nn.functional.interpolate(img[None], size=(TAMANHO, TAMANHO), mode="bilinear")
ref = modelo_cpu(list(img))[0]
caixas, pontuacoes, rotulos = ort.InferenceSession(str(saida / "objetos.onnx")).run(None, {"imagem": img.numpy()})
print("caixas PyTorch:", len(ref["boxes"]), "| ONNX:", len(caixas))
if len(caixas):
    # Compara as pontuações ordenadas (caixas empatadas podem vir em outra ordem) e a caixa mais confiável
    dif_pont = np.abs(np.sort(ref["scores"].detach().numpy()) - np.sort(pontuacoes)).max()
    dif_caixa = np.abs(ref["boxes"][0].detach().numpy() - caixas[0]).max()
    print(f"diferença nas pontuações: {dif_pont:.2e} | na caixa principal: {dif_caixa:.2f} px")
print("tamanho:", (saida / "objetos.onnx").stat().st_size / 1e6, "MB ->", saida)